Cell 1: 导入依赖与路径配置

In [ ]:
# ==========================================
# Cell 1: 基础依赖与配置 (Advance V1.0.0 - 低效站点剔除)
# ==========================================
import os, pathlib, json, math, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# 配置
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ROOT = pathlib.Path('.')

# 本次进阶训练的输出路径
OUT_DIR = ROOT / 'output_100'
CM_DIR = OUT_DIR / 'confusion_matrices'
MODEL_DIR = OUT_DIR / 'models'
CM_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# 随机种子 - 保证可复现性
def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
print(f'Device: {DEVICE}')
print(f'Output Base Dir: {OUT_DIR}')

Cell 2: 解析预测结果，生成站点“白名单”


在这个核心单元格中，我们直接根据先前的分类准确度筛选特征

In [ ]:
# ==========================================
# Cell 2: 站点评估与筛选 (Feature Selection < 50%)
# ==========================================
PRED_CSV_PATH = ROOT / 'confusion_matrices224' / 'predictions_test_station_day_v224.csv'
stations_list_path = ROOT / 'data' / 'stations_list'

# 1. 解析原始站点的总列表
station_ids = [int(x) for x in stations_list_path.read_text(encoding='utf-8', errors='ignore').strip().strip('[]').split() if x.isdigit()]
TOTAL_STATIONS = len(station_ids)

# 2. 读取上一版的预测结果
print("正在加载历史预测记录并进行准确率分析...")
df_preds = pd.read_csv(PRED_CSV_PATH)

# 计算每个站点 (station_pos) 的平均预测准确率
station_acc = df_preds.groupby('station_pos')['correct'].mean().reset_index()
station_acc.rename(columns={'correct': 'accuracy'}, inplace=True)

# 3. 设定阈值：找出 >= 50% 的站点
threshold = 0.5
keep_stations = station_acc[station_acc['accuracy'] >= threshold]
keep_positions = keep_stations['station_pos'].values

# 4. 生成全局保留掩码 (Global Keep Mask)
global_keep_mask = np.zeros(TOTAL_STATIONS, dtype=bool)
global_keep_mask[keep_positions] = True

print(f"--- 站点筛选统计 ---")
print(f"初始站点总数: {TOTAL_STATIONS}")
print(f"保留站点数量 (>=50%): {len(keep_positions)}")
print(f"剔除站点数量 (<50%): {TOTAL_STATIONS - len(keep_positions)}")

# 保存保留列表，方便查阅
np.save(OUT_DIR / 'keep_station_indices.npy', keep_positions)

Cell 3: 数据加载与多维矩阵切片

这一步将对原始数据集进行“瘦身”。请将下面代码中的数据加载部分替换为你具体的读取逻辑（例如直接读取 txt 或 io.loadmat）。

In [ ]:
# ==========================================
# Cell 3: 数据读取、预处理与增强
# ==========================================
train_path = ROOT / 'data' / 'PEMS_train'
test_path  = ROOT / 'data' / 'PEMS_test'
labels_train_path = ROOT / 'data' / 'PEMS_trainlabels'
labels_test_path  = ROOT / 'data' / 'PEMS_testlabels'
groups_index_path = ROOT / 'data' / 'processed' / 'pems_sf_groups_index.json'

def parse_day_matrix(line: str):
    line = line.strip()
    if line.startswith('[') and line.endswith(']'): line = line[1:-1]
    rows = [r.strip() for r in line.split(';') if r.strip()]
    data = []
    for r in rows:
        nums = [float(x) for x in r.split() if x.replace('.', '', 1).isdigit()]
        if nums: data.append(nums)
    if not data: return None
    arr = np.full((len(data), max(len(rr) for rr in data)), np.nan, dtype=float)
    for i, rr in enumerate(data): arr[i, :len(rr)] = rr
    return arr

def iter_day_matrices(path: pathlib.Path, limit=None):
    with path.open('r', encoding='utf-8', errors='ignore') as f:
        for i, line in enumerate(f):
            if limit is not None and i >= limit: break
            mat = parse_day_matrix(line)
            yield mat if mat is not None else None

def load_labels(path: pathlib.Path):
    txt = path.read_text(encoding='utf-8', errors='ignore').strip().strip('[]')
    return np.array([int(x) for x in txt.split() if x.isdigit()], dtype=int)

labels_train = load_labels(labels_train_path)
labels_test  = load_labels(labels_test_path)

# 加载原始分组掩码，并与我们的【全局过滤掩码】做 AND 运算
group_station_masks = {}
if groups_index_path.exists():
    processed = json.loads(groups_index_path.read_text(encoding='utf-8'))
    groups_index = processed.get('groups_index', {})
    sid_to_pos = {sid: i for i, sid in enumerate(station_ids)}
    for g in ['g1','g2','g3','g4','g5']:
        mask = np.zeros(len(station_ids), dtype=bool)
        for sid in groups_index.get(g, []):
            if sid in sid_to_pos: mask[sid_to_pos[sid]] = True
        # 【核心改动】：交集过滤，剔除<50%的站点
        group_station_masks[g] = mask & global_keep_mask
    print('分组掩码加载完毕，并已应用准确率过滤。')
else:
    print('警告：未找到分组文件。')

def _process_curve(curve: np.ndarray):
    curve = curve[:144]
    if curve.shape[0] < 144:
        curve = np.pad(curve, (0, 144 - curve.shape[0]), mode='constant', constant_values=np.nan)
    idx = np.arange(curve.size)
    mask = ~np.isnan(curve)
    if mask.any(): curve = np.interp(idx, idx[mask], curve[mask])
    curve = np.nan_to_num(curve, nan=0.0)

    min_val, max_val = curve.min(), curve.max()
    denom = max_val - min_val + 1e-8
    norm_curve = (curve - min_val) / denom
    
    mean_val = np.mean(norm_curve)
    std_val  = np.std(norm_curve)
    log_max_val = np.log1p(max_val) / 10.0 
    stats = np.array([mean_val, std_val, log_max_val], dtype=np.float32)

    grad_curve = np.gradient(norm_curve)
    result = np.stack([norm_curve, grad_curve], axis=0)
    return result.astype(np.float32), stats

class PEMS_CNNDataset(Dataset):
    def __init__(self, split_path: pathlib.Path, labels: np.ndarray, use_mask: np.ndarray, station_ids: list, augment=False, split_name=""):
        self.samples = []
        self.augment = augment
        self.split_name = split_name
        station_ids_arr = np.asarray(station_ids, dtype=int)

        if use_mask is not None and use_mask.shape[0] == station_ids_arr.shape[0]:
            kept_station_ids = station_ids_arr[use_mask]
            kept_station_pos = np.nonzero(use_mask)[0]
        else:
            kept_station_ids = station_ids_arr
            kept_station_pos = np.arange(station_ids_arr.shape[0], dtype=int)

        for day_i, mat in enumerate(iter_day_matrices(split_path)):
            if day_i >= len(labels): break
            if mat is None or mat.shape[1] < 144: continue
            y = int(labels[day_i]) - 1
            if y < 0 or y > 6: continue

            if use_mask is not None and use_mask.shape[0] == mat.shape[0]:
                mat = mat[use_mask, :]

            if mat.shape[0] != kept_station_ids.shape[0]: continue

            for local_sidx in range(mat.shape[0]):
                two_channel_seq, stats = _process_curve(mat[local_sidx, :144])
                if not np.isfinite(two_channel_seq).all(): continue
                meta = {
                    "station_id": int(kept_station_ids[local_sidx]),
                    "station_pos": int(kept_station_pos[local_sidx]),
                    "day_i": int(day_i),
                    "split": self.split_name,
                }
                self.samples.append((two_channel_seq, stats, y, meta))

        self.n = len(self.samples)

    def __len__(self): return self.n

    def __getitem__(self, idx):
        seq, stats, y, meta = self.samples[idx]
        seq_tensor = torch.from_numpy(seq)
        if self.augment:
            scale = random.uniform(0.95, 1.05)
            seq_tensor = seq_tensor * scale
            shift_steps = random.randint(-2, 2)
            if shift_steps != 0: seq_tensor = torch.roll(seq_tensor, shifts=shift_steps, dims=1)
            noise = torch.randn_like(seq_tensor) * 0.005
            seq_tensor = seq_tensor + noise
        return seq_tensor, torch.from_numpy(stats), torch.tensor(y, dtype=torch.long), meta

Cell 4: 模型初始化与重新训练

重点注意：咱们新模型的输入特征维度不再是 963，而是 len(keep_indices)。

In [ ]:
# ==========================================
# Cell 4: 构建DataLoader
# ==========================================
group_loaders = {}
for g, mask in group_station_masks.items():
    print(f'准备分组 {g} 数据 (经过准确率过滤后)...')
    
    ds_train_source = PEMS_CNNDataset(train_path, labels_train, mask, station_ids=station_ids, augment=True, split_name="train")
    ds_val_source = PEMS_CNNDataset(train_path, labels_train, mask, station_ids=station_ids, augment=False, split_name="val")
    ds_test = PEMS_CNNDataset(test_path, labels_test, mask, station_ids=station_ids, augment=False, split_name="test")
    
    total_len = len(ds_train_source)
    n_train = int(total_len * 0.8)
    indices = torch.randperm(total_len, generator=torch.Generator().manual_seed(42)).tolist()
    train_indices = indices[:n_train]
    val_indices = indices[n_train:]
    
    train_subset = Subset(ds_train_source, train_indices)
    val_subset   = Subset(ds_val_source, val_indices)
    
    group_loaders[g] = {
        'train': DataLoader(train_subset, batch_size=64, shuffle=True, num_workers=0),
        'val':   DataLoader(val_subset, batch_size=128, shuffle=False, num_workers=0),
        'test':  DataLoader(ds_test, batch_size=128, shuffle=False, num_workers=0),
    }
    print(f'  - Train: {len(train_subset)} (Augmented), Val: {len(val_subset)} (Clean), Test: {len(ds_test)}')

Cell 5: 模型架构定义

In [ ]:
# ==========================================
# Cell 5: 1D-CNN + SE + Fusion
# ==========================================
class SEBlock(nn.Module):
    def __init__(self, channel, reduction=16):
        super(SEBlock, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(channel, channel // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channel // reduction, channel, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        b, c, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1)
        return x * y.expand_as(x)

class Simple1DCNN(nn.Module):
    def __init__(self, in_channels=2, num_classes=7):
        super(Simple1DCNN, self).__init__()
        self.block1 = nn.Sequential(
            nn.Conv1d(in_channels, 32, 7, padding=3),
            nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2)
        )
        self.se1 = SEBlock(32, reduction=4) 
        self.drop1 = nn.Dropout1d(p=0.1) 

        self.block2 = nn.Sequential(
            nn.Conv1d(32, 64, 5, padding=2),
            nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2)
        )
        self.se2 = SEBlock(64, reduction=8)
        self.drop2 = nn.Dropout1d(p=0.1)

        self.block3 = nn.Sequential(
            nn.Conv1d(64, 128, 3, padding=1),
            nn.BatchNorm1d(128), nn.ReLU()
        )
        self.se3 = SEBlock(128, reduction=16)

        self.gap = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Dropout(0.25), 
            nn.Linear(131, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )
        
    def forward(self, x, stats):
        x = self.block1(x); x = self.se1(x); x = self.drop1(x)
        x = self.block2(x); x = self.se2(x); x = self.drop2(x)
        x = self.block3(x); x = self.se3(x)
        x = self.gap(x).view(x.size(0), -1) 
        combined = torch.cat([x, stats], dim=1) 
        logits = self.fc(combined)
        return logits

Cell 6: 训练循环与执行

In [ ]:
# ==========================================
# Cell 6: 训练循环
# ==========================================
def train_model(loaders, group_name, epochs=60):
    model = Simple1DCNN(in_channels=2, num_classes=7).to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-3) 
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    
    weights = torch.tensor([1.0, 2.0, 4.0, 2.0, 1.5, 1.0, 1.0]).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.1)
    
    best_acc = 0.0
    best_state = None
    patience = 15 
    counter = 0
    
    for epoch in range(1, epochs+1):
        model.train()
        total_loss = 0
        for x, stats, y, _meta in loaders['train']:
            x, stats, y = x.to(DEVICE), stats.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            out = model(x, stats)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            
        scheduler.step() 
        
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for x, stats, y, _meta in loaders['val']:
                x, stats, y = x.to(DEVICE), stats.to(DEVICE), y.to(DEVICE)
                out = model(x, stats)
                preds = torch.argmax(out, dim=1)
                correct += (preds == y).sum().item()
                total += y.size(0)
        
        val_acc = correct / total if total > 0 else 0
        
        if val_acc > best_acc:
            best_acc = val_acc
            best_state = model.state_dict()
            counter = 0
        else:
            counter += 1
            
        if epoch % 5 == 0:
            current_lr = scheduler.get_last_lr()[0]
            print(f"[{group_name}] Epoch {epoch}/{epochs} | Loss: {total_loss/len(loaders['train']):.4f} | Val Acc: {val_acc:.2%} | LR: {current_lr:.6f}")
        
        if counter >= patience:
            print(f"[{group_name}] Early stopping at epoch {epoch}")
            break
            
    return best_state, best_acc

group_models = {}
for g, loaders in group_loaders.items():
    print(f"\n>>> 开始训练分组: {g}")
    state, acc = train_model(loaders, g, epochs=60) 
    group_models[g] = state
    print(f"[{g}] 训练完成，最佳验证准确率: {acc:.2%}")
    if state:
        torch.save(state, MODEL_DIR / f'cnn_v100_{g}.pth')

print("\n所有进阶模型训练结束。")

Cell 7: 评估与导出

In [ ]:
# ==========================================
# Cell 7: 评估与结果导出
# ==========================================
def get_predictions(model_state, loader):
    model = Simple1DCNN(in_channels=2, num_classes=7).to(DEVICE)
    model.load_state_dict(model_state)
    model.eval()
    y_true, y_pred, metas = [], [], []

    with torch.no_grad():
        for x, stats, y, meta in loader:
            x, stats = x.to(DEVICE), stats.to(DEVICE)
            logits = model(x, stats)
            preds = torch.argmax(logits, dim=1)
            y_true.extend(y.cpu().numpy().tolist())
            y_pred.extend(preds.cpu().numpy().tolist())

            if isinstance(meta, list):
                metas.extend(meta)
            elif isinstance(meta, dict):
                bs = len(y)
                for i in range(bs):
                    metas.append({k: meta[k][i] for k in meta})
            else:
                metas.extend(list(meta))
    return np.array(y_true), np.array(y_pred), metas

LABELS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

def export_predictions_station_day(mode="test"):
    rows = []
    for g in ["g1","g2","g3","g4","g5"]:
        if g not in group_models: continue
        y_t, y_p, metas = get_predictions(group_models[g], group_loaders[g][mode])

        for i in range(len(y_t)):
            m = metas[i]
            yt, yp = int(y_t[i]), int(y_p[i])
            rows.append({
                "group": g, "split": mode,
                "station_id": int(m.get("station_id", -1)),
                "day_i": int(m.get("day_i", -1)),         
                "station_pos": int(m.get("station_pos", -1)),
                "y_true": yt, "y_pred": yp,
                "true_label": LABELS[yt], "pred_label": LABELS[yp],
                "correct": (yt == yp),
            })

    df = pd.DataFrame(rows).sort_values(
        ["correct","group","station_id","day_i"], ascending=[True, True, True, True]
    ).reset_index(drop=True)

    out_all = CM_DIR / f"predictions_{mode}_station_day_v100.csv"
    out_bad = CM_DIR / f"misclassified_{mode}_station_day_v100.csv"
    df.to_csv(out_all, index=False, encoding="utf-8-sig")
    df[df["correct"] == False].to_csv(out_bad, index=False, encoding="utf-8-sig")

    print("ALL :", out_all)
    print("BAD :", out_bad)
    print(f"Total={len(df)} Bad={(~df['correct']).sum()} Acc={df['correct'].mean():.2%}")
    return df

df_test = export_predictions_station_day("test")